# 01 Synthetic Prediction Market Signal

This notebook demonstrates the public version of the prediction-market research workflow using synthetic data only.

The tables imitate two asynchronous markets:

- an underlying financial market observed every minute
- a prediction-market contract observed less frequently

The goal is to build a timestamp-safe decision table, create rolling features, form placeholder signals, and run a toy spread-aware backtest.

No private data, real market identifiers, strategy thresholds, sizing rules, or live performance appear in this notebook.

## Step 0: Imports And Paths

This cell loads pandas, plotting tools, and the small reusable modules in `src/`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

from data.synthetic_data import make_synthetic_market_data
from data.timestamp_alignment import align_prediction_to_underlying
from features.rolling_features import add_rolling_zscore
from signals.placeholder_signals import symmetric_threshold_signal, capped_position_from_signal
from backtesting.event_backtester import run_position_backtest, summarise_backtest

## Step 1: Generate Synthetic Market Tables

`underlying` is the faster market. `prediction` is the slower market. The slower update schedule makes stale-price handling visible.

In [ ]:
underlying, prediction = make_synthetic_market_data(
    seed=7,
    periods=360,
)

print("underlying rows:", len(underlying))
print("prediction rows:", len(prediction))

display(underlying.head())
display(prediction.head())

## Step 2: Align The Latest Observable Prediction Price

A backward as-of merge gives each underlying timestamp the most recent prediction-market observation available at that time. The stale-minute column is kept in the table rather than hidden.

In [ ]:
aligned = align_prediction_to_underlying(
    underlying,
    prediction,
    max_stale_minutes=20,
)

aligned = aligned.sort_values("timestamp_utc").reset_index(drop=True)

alignment_audit = {
    "rows": len(aligned),
    "missing_prediction_price": aligned["prediction_price"].isna().sum(),
    "max_stale_minutes": aligned["prediction_stale_minutes"].max(),
}

display(pd.DataFrame([alignment_audit]))
display(aligned.head(12))

## Step 3: Rolling Feature Scaffold

The feature is intentionally generic: a rolling z-score of the underlying return. The signal rule below is a placeholder, not a trading rule.

In [ ]:
FEATURE_WINDOW = 30
PLACEHOLDER_Z_THRESHOLD = 1.25
MAX_TOY_POSITION = 1.0

signal_table = add_rolling_zscore(
    aligned,
    column="underlying_return",
    window=FEATURE_WINDOW,
    min_periods=FEATURE_WINDOW,
)

z_col = f"underlying_return_zscore_{FEATURE_WINDOW}"
signal_table["placeholder_signal"] = symmetric_threshold_signal(
    signal_table,
    zscore_col=z_col,
    threshold=PLACEHOLDER_Z_THRESHOLD,
)

signal_table["toy_position"] = capped_position_from_signal(
    signal_table["placeholder_signal"],
    max_position=MAX_TOY_POSITION,
)

missing_price = signal_table["prediction_price"].isna()
signal_table.loc[missing_price, ["placeholder_signal", "toy_position"]] = 0

display(signal_table[[
    "timestamp_utc",
    "underlying_return",
    z_col,
    "prediction_price",
    "prediction_stale_minutes",
    "placeholder_signal",
    "toy_position",
]].tail(12))

## Step 4: Toy Backtest

The backtester shifts the held position by one row before applying returns. That keeps the signal timestamp separate from the price movement being measured.

In [ ]:
backtest_input = signal_table.dropna(subset=["prediction_price"]).copy()

backtest = run_position_backtest(
    backtest_input,
    price_col="prediction_price",
    position_col="toy_position",
    cost_rate=0.0005,
)

summary = summarise_backtest(backtest)
summary_df = pd.DataFrame([summary])

display(summary_df)
display(backtest[[
    "timestamp_utc",
    "prediction_price",
    "toy_position",
    "held_position",
    "gross_pnl",
    "cost",
    "net_pnl",
    "equity_curve",
]].tail(10))

## Step 5: Visual Check

The plots show the synthetic underlying market, prediction-market probability, placeholder position, and toy equity curve.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)

axes[0].plot(backtest["timestamp_utc"], backtest["underlying_price"])
axes[0].set_title("Synthetic underlying price")

axes[1].plot(backtest["timestamp_utc"], backtest["prediction_price"])
axes[1].set_title("Synthetic prediction-market probability")

axes[2].step(backtest["timestamp_utc"], backtest["toy_position"], where="post")
axes[2].set_title("Placeholder position")

axes[3].plot(backtest["timestamp_utc"], backtest["equity_curve"])
axes[3].set_title("Toy net P&L curve")

for ax in axes:
    ax.grid(True, alpha=0.25)

fig.tight_layout()
plt.show()

## Public Version Omissions

This notebook excludes real markets, exact thresholds, sizing rules, token identifiers, private datasets, and private performance. The signal is deliberately simple so the code demonstrates research mechanics rather than a deployable strategy.